## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-24-23-55-01 +0000,bbc,Thomas Markle indicted by federal grand jury i...,https://www.bbc.co.uk/news/articles/crwy4egxn5...
1,2026-11-24-23-55-00 +0000,nyt,Prosecutor Urged Justice Dept. to Reconsider C...,https://www.nytimes.com/2026/11/24/us/politics...
2,2026-11-24-23-52-35 +0000,nyt,"Abrego Garcia Will Voluntarily Leave U.S., End...",https://www.nytimes.com/2026/11/24/us/politics...
3,2026-11-24-23-50-56 +0000,nypost,Hero dad drowns rescuing kids from current whi...,https://nypost.com/2026/11/24/world-news/hero-...
4,2026-11-24-23-50-06 +0000,startribune,Asian grocer H Mart will open in St. Paul’s Mi...,https://www.startribune.com/h-mart-mn-st-paul-...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2744/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
125,nvidia,43
62,trump,37
265,huang,33
264,jensen,25
22,will,15


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
352,2026-11-24-08-52-18 +0000,nypost,Trump shrugs off GOP concerns that Jensen Huan...,https://nypost.com/2026/11/24/us-news/trump-sh...,190
91,2026-11-24-21-49-00 +0000,wsj,Jensen Huang will face a bipartisan maelstrom ...,https://www.wsj.com/politics/policy/jensen-hua...,177
177,2026-11-24-18-28-12 +0000,nypost,Trump says cheating Nvidia CEO Jensen Huang an...,https://nypost.com/2026/11/24/us-news/trump-sa...,167
315,2026-11-24-10-53-37 +0000,bbc,Trump faces backlash after saying he 'convince...,https://www.bbc.co.uk/news/articles/cq5y1k8ez4...,166
282,2026-11-24-12-00-00 +0000,wsj,Secretary of State Marco Rubio sounded alarms ...,https://www.wsj.com/politics/policy/rubio-soun...,163


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
352,190,2026-11-24-08-52-18 +0000,nypost,Trump shrugs off GOP concerns that Jensen Huan...,https://nypost.com/2026/11/24/us-news/trump-sh...
91,76,2026-11-24-21-49-00 +0000,wsj,Jensen Huang will face a bipartisan maelstrom ...,https://www.wsj.com/politics/policy/jensen-hua...
185,54,2026-11-24-18-16-39 +0000,cbc,Rudy Giuliani awarded Presidential Medal of Fr...,https://www.cbc.ca/news/world/us-medal-of-free...
371,50,2026-11-24-03-30-00 +0000,nypost,China’s second-richest man’s deal to drain Cal...,https://nypost.com/2026/11/23/us-news/chinese-...
292,46,2026-11-24-11-28-30 +0000,nypost,Two suspects arrested after break-in reported ...,https://nypost.com/2026/11/24/world-news/princ...
367,44,2026-11-24-04-15-00 +0000,wsj,A federal appeals court has cleared the way fo...,https://www.wsj.com/politics/national-security...
89,40,2026-11-24-21-49-30 +0000,nypost,Jury finds Chinese tech giant Huawei guilty of...,https://nypost.com/2026/11/24/business/jury-fi...
140,38,2026-11-24-20-28-47 +0000,chitri,Residents plead with Lake County Board to help...,https://www.chicagotribune.com/2026/11/24/lake...
339,37,2026-11-24-09-53-51 +0000,wsj,Stock Market Today: Nasdaq Resumes Selloff as ...,https://www.wsj.com/livecoverage/stock-market-...
354,36,2026-11-24-08-02-54 +0000,nypost,Pope Francis saw ‘gates of hell’ during church...,https://nypost.com/2026/11/24/world-news/pope-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
